# NLP, Tokenización y Vectorización

### UTN-FRSF · Ingeniería en Sistemas de Información · Inteligencia Artificial
**Dr. Jorge Roa · Dra. María de los Milagros Gutiérrez**

---

*¿Cómo convierte una máquina texto en algo con lo que puede operar?*

## El mapa del curso

| Clase | Tema | Pregunta central |
|---|---|---|
| **1** | NLP, Tokenización, Embeddings | ¿Cómo representa texto una máquina? |
| 2 | LLMs | ¿Cómo genera texto un modelo? |
| 3 | RAG | ¿Cómo le doy conocimiento propio? |
| 4 | Agentes | ¿Cómo actúa de forma autónoma? |

🎯 **TP integrador:** agente RAG funcional sobre un dominio a elección

## El pipeline de un LLM — caja negra

```
  Texto         ✂️ Tokenización    🧭 Embeddings      ⚙️ Modelo          💬 Output
  de entrada  ──────────────────────────────────────────────────────────────────▶
  (hoy)           (hoy)              (hoy)             (Clase 2)         (Clase 2)
```

> **Cada clase de este curso abre una o más cajas de este pipeline.**

## ⚙️ Setup — ejecutar antes de arrancar

```bash
pip install nltk spacy tiktoken sentence-transformers \
            umap-learn scikit-learn numpy pandas matplotlib
python -m spacy download es_core_news_sm
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams = plt.rcParams
plt.rcParams['figure.dpi'] = 120

# Corpus de la clase — 20 oraciones sobre IA y sistemas inteligentes
CORPUS = [
    # NLP y lenguaje
    "Los modelos de lenguaje procesan texto para entender su significado.",
    "El procesamiento del lenguaje natural permite a las máquinas leer y escribir.",
    "Tokenizar un texto significa dividirlo en unidades mínimas de significado.",
    "Los embeddings representan palabras como vectores en un espacio matemático.",
    "La similitud semántica entre palabras puede medirse con la distancia coseno.",
    # LLMs y modelos
    "Los modelos de lenguaje de gran escala se entrenan con enormes volúmenes de texto.",
    "GPT y Llama son ejemplos de modelos generativos basados en la arquitectura Transformer.",
    "El fine-tuning adapta un modelo preentrenado a una tarea específica.",
    "Un modelo base predice el siguiente token; un modelo instruct sigue instrucciones.",
    "El tamaño del contexto limita cuánto texto puede procesar un modelo a la vez.",
    # RAG y recuperación
    "RAG combina recuperación de información con generación de texto.",
    "Un vector store guarda embeddings para hacer búsquedas semánticas eficientes.",
    "La búsqueda híbrida combina similitud semántica con búsqueda por palabras clave.",
    "El chunking divide documentos largos en fragmentos aptos para ser indexados.",
    "Recuperar contexto relevante reduce las alucinaciones del modelo.",
    # Agentes
    "Un agente inteligente percibe su entorno y toma acciones para alcanzar objetivos.",
    "Los agentes modernos usan LLMs para razonar y decidir qué herramienta usar.",
    "El patrón ReAct alterna entre razonamiento y acción en un ciclo iterativo.",
    "Los sistemas multiagente dividen tareas complejas entre agentes especializados.",
    "Claude Code y Manus son ejemplos de agentes profundos con planificación y memoria.",
]

CATEGORIAS = ['NLP'] * 5 + ['LLMs'] * 5 + ['RAG'] * 5 + ['Agentes'] * 5
print(f'✓ Corpus cargado: {len(CORPUS)} oraciones')

# Bloque 2
## Representaciones clásicas — BoW y TF-IDF

---

*¿Cómo representaba el texto una máquina antes de los embeddings?*

## El problema de fondo

### ¿Cómo le explicás a una computadora que "auto" y "coche" significan lo mismo?

| Problema | Ejemplo | ¿Lo resuelve BoW? |
|---|---|---|
| **Sinonimia** | "auto" y "coche" son palabras distintas pero equivalentes | ✗ No |
| **Polisemia** | "banco" puede ser institución o mueble | ✗ No |
| **Contexto** | "el banco quebró" vs "me senté en el banco" | ✗ No |

## Bag of Words — la primera solución

**Idea:** representar cada documento como un vector de conteos de palabras.

```
Doc 1: "El gato come pescado"
Doc 2: "El perro come carne"
Doc 3: "El gato duerme"

        carne  come  duerme  el  gato  perro  pescado
Doc 1:    0      1      0     1    1      0       1
Doc 2:    1      1      0     1    0      1       0
Doc 3:    0      0      1     1    1      0       0
```

⚠️ **Problema:** "auto" y "coche" serían columnas distintas sin ninguna relación.  
⚠️ **Problema:** el orden de las palabras se pierde completamente.

> → BM25 mejora BoW para búsqueda — lo usamos en Clase 3 dentro de **hybrid search en RAG**.

> 📖 *Jurafsky, D. & Martin, J. H. (2024). Speech and Language Processing, Cap. 6. https://web.stanford.edu/~jurafsky/slp3/6.pdf*

## TF-IDF — pesando palabras por relevancia

**Problema de BoW:** trata todas las palabras igual. "el", "la", "de" reciben el mismo peso que "tokenización".

**Idea:** una palabra que aparece en **todos** los documentos dice poco. Una que aparece en **uno solo** dice mucho.

```
  TF-IDF(término, documento) = TF × IDF

  TF  = frecuencia del término en el documento
        (cuántas veces aparece "gato" en Doc 1)

  IDF = log( N / cantidad de docs que contienen el término )
        (si "el" aparece en los 3 docs: IDF bajo → peso bajo)
        (si "pescado" aparece en 1 de 3: IDF alto → peso alto)
```

| Término | Aparece en | TF-IDF |
|---|---|---|
| "el" | Todos los docs | **muy bajo** — no distingue |
| "come" | 2 de 3 docs | **bajo** |
| "gato" | 2 de 3 docs | **medio** |
| "pescado" | 1 de 3 docs | **alto** — distingue Doc 1 |

> → BM25 es una mejora de TF-IDF para búsqueda — lo usamos en Clase 3 en **hybrid search**.

> 📖 *Jurafsky, D. & Martin, J. H. (2024). Speech and Language Processing, Cap. 6. https://web.stanford.edu/~jurafsky/slp3/6.pdf*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

oraciones_ejemplo = [
    "El gato come pescado",
    "El perro come carne",
    "El gato duerme",
]

# Bag of Words
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(oraciones_ejemplo)

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(3)]
)
print('Matriz Bag of Words:')
print(df_bow.to_string())

In [ ]:
# TF-IDF — pesa las palabras por rareza
tfidf = TfidfVectorizer()
matriz_tfidf = tfidf.fit_transform(oraciones_ejemplo)

df_tfidf = pd.DataFrame(
    matriz_tfidf.toarray().round(3),
    columns=tfidf.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(3)]
)
print('Matriz TF-IDF:')
print(df_tfidf.to_string())
print()
print('Observar: "el" y "come" → peso bajo (aparecen en varios docs)')
print('          "pescado", "carne", "duerme" → peso alto (aparecen en uno solo)')

In [ ]:
# ✏️ EJERCICIO — construí tu propia matriz BoW
# Reemplazá las oraciones por las tuyas
# ¿Qué pasa si usás sinónimos? ¿Y palabras con varios significados?

mis_oraciones = [
    # "...",
    # "...",
]

# Completá el código acá
# vec = CountVectorizer()
# ...

# Bloque 3
## Tokenización

---

*¿Cómo parte el texto un LLM real?*

## ¿Qué es un token?

### ¿Cómo partirías "tokenización" si nunca la hubieras visto?

**Por palabras** → `["tokenización"]`
✓ Simple &nbsp;&nbsp; ✗ Falla con palabras nuevas — vocabulario infinito

---

**Por caracteres** → `["t", "o", "k", "e", "n", "i", "z", "a", "c", "i", "ó", "n"]`
✓ Nunca falla &nbsp;&nbsp; ✗ Secuencias muy largas, pierde significado

---

**Por subpalabras ✓** → `["token", "ización"]` — *el balance ideal*
✓ Vocabulario finito &nbsp;&nbsp; ✓ Maneja palabras nuevas &nbsp;&nbsp; ✗ Requiere entrenamiento previo

> 📖 *Sennrich, R., Haddow, B. & Birch, A. (2016). Neural Machine Translation of Rare Words with Subword Units. ACL 2016. https://arxiv.org/abs/1508.07909*

## BPE, WordPiece, SentencePiece

| Algoritmo | Modelos | Ejemplo |
|---|---|---|
| **BPE** | GPT-4o, Llama, Mistral | `token` + `ización` |
| **WordPiece** | BERT, DistilBERT | `token` + `##ización` |
| **SentencePiece** | Llama 2, T5, mT5 | `▁token` + `ización` |

**BPE** — fusiona el par de caracteres más frecuente iterativamente  
**WordPiece** — maximiza la probabilidad del corpus (prefija subpalabras con `##`)  
**SentencePiece** — no pre-tokeniza por espacios, ideal para idiomas sin separadores

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize
import spacy

oracion = "La tokenización es fundamental para el procesamiento del lenguaje natural."

# NLTK
tokens_nltk = word_tokenize(oracion, language='spanish')
print(f'NLTK ({len(tokens_nltk)} tokens):')
print(f'  {tokens_nltk}')

# spaCy (lemas + POS tags)
nlp = spacy.load('es_core_news_sm')
doc = nlp(oracion)
print(f'\nspaCy ({len(doc)} tokens):')
for token in doc:
    print(f'  {token.text:<18} lema: {token.lemma_:<18} POS: {token.pos_}')

In [ ]:
# BPE implementado desde cero — el algoritmo expuesto
from collections import Counter

def get_pairs(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(pair, vocab):
    nuevo = {}
    bigram, reemplazo = ' '.join(pair), ''.join(pair)
    for word, freq in vocab.items():
        nuevo[word.replace(bigram, reemplazo)] = freq
    return nuevo

vocab = {
    'b a j o </w>': 5, 'b a l a </w>': 3,
    'b a s e </w>': 4, 'c a s a </w>': 6,
    'c a s o </w>': 2, 's a l a </w>': 3,
}

print('--- Iteraciones BPE ---')
for i in range(5):
    pairs = get_pairs(vocab)
    mejor = pairs.most_common(1)[0][0]
    vocab = merge_pair(mejor, vocab)
    print(f'Iter {i+1}: fusionar {mejor} → {"" .join(mejor)}')
    print(f'  vocab: {list(vocab.keys())}')

In [ ]:
import tiktoken

enc = tiktoken.get_encoding('o200k_base')  # tokenizer GPT-4o — sin API key

frase_es = "La inteligencia artificial está transformando la ingeniería de software moderna."
frase_en = "Artificial intelligence is transforming modern software engineering."

tokens_es = enc.encode(frase_es)
tokens_en = enc.encode(frase_en)

print('Tokenizer GPT-4o (o200k_base) — corre local, sin API key')
print()
print(f'🇦🇷  Español: {len(tokens_es)} tokens')
print(f'    {[enc.decode([t]) for t in tokens_es]}')
print()
print(f'🇺🇸  Inglés:  {len(tokens_en)} tokens')
print(f'    {[enc.decode([t]) for t in tokens_en]}')
print()
print(f'💰  Diferencia: {len(tokens_es) - len(tokens_en)} tokens más en español')
print(f'    Si usás una API de pago, escribir en español puede costarte más.')

In [ ]:
# ✏️ EJERCICIO — experimentá con el tokenizer
# 1. Probá con una frase tuya en español e inglés
# 2. ¿Qué pasa con palabras técnicas como "microservicio" o "tokenización"?
# 3. ¿Qué pasa con emojis o caracteres especiales?

mi_frase = "..."  # reemplazá

# Completá acá
# tokens = enc.encode(mi_frase)
# print(f'{len(tokens)} tokens: {[enc.decode([t]) for t in tokens]}')

# Bloque 4
## Embeddings y similitud semántica

---

*¿Cómo representamos significado, no solo palabras?*

## Embeddings — representar significado como vector

**¿Recuerdan el problema de BoW?** "auto" y "coche" eran columnas distintas sin relación.

Los embeddings lo resuelven:

- **Palabras con contextos similares → vectores cercanos**  
  "auto" y "coche" aparecen cerca de "manejar", "rueda", "velocidad" → sus vectores quedan cerca

- **La distancia codifica significado**  
  "auto" está lejos de "banana" — tienen nada en común semánticamente

- **Word2Vec — limitación ⚠️**  
  Cada palabra tiene UN solo vector: `"banco"` (dinero) = `"banco"` (mueble) → mismo punto

> **Sentence Transformers** resuelven esto — generan vectores que cambian según el contexto completo de la oración.

> 📖 *Mikolov, T., et al. (2013). Efficient Estimation of Word Representations in Vector Space. https://arxiv.org/abs/1301.3781*  
> 📖 *Reimers, N. & Gurevych, I. (2019). Sentence-BERT. EMNLP 2019. https://arxiv.org/abs/1908.10084*

In [ ]:
from sentence_transformers import SentenceTransformer

# Modelo multilingüe liviano (~90MB) — funciona bien en español
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print('Generando embeddings del corpus...')
embeddings = modelo.encode(CORPUS, show_progress_bar=True)

print(f'\n✓ Shape: {embeddings.shape}  →  {len(CORPUS)} oraciones × {embeddings.shape[1]} dimensiones')
print(f'Primeros 6 valores del embedding [0]: {embeddings[0][:6].round(4)}')

# Guardar para Clase 3 (RAG)
np.save('corpus_embeddings_clase1.npy', embeddings)
pd.DataFrame({'oracion': CORPUS, 'categoria': CATEGORIAS}).to_csv('corpus_clase1.csv', index=False)
print('\n✓ Guardado: corpus_embeddings_clase1.npy  →  lo usamos en Clase 3')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Los tres pares de la slide — verificamos los scores en vivo
pares = [
    ("Me gusta el fútbol", "Disfruto el fútbol"),
    ("Me gusta el fútbol", "Juego al tenis"),
    ("Me gusta el fútbol", "La física cuántica es compleja"),
]

print('Similitud coseno entre pares de oraciones:')
print('-' * 60)
for a, b in pares:
    score = cosine_similarity(modelo.encode([a]), modelo.encode([b]))[0][0]
    barra = '█' * int(score * 25)
    print(f'\n  A: "{a}"')
    print(f'  B: "{b}"')
    print(f'  Score: {score:.4f}  {barra}')

In [ ]:
# Visualización 2D de embeddings — espacio semántico con matplotlib
# Representación didáctica: vectores 2D que ilustran relaciones semánticas

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

# Coordenadas 2D representativas (simplificadas para ilustración)
# Agrupadas semánticamente como lo haría un modelo real
palabras = {
    # Cluster vehículos
    'auto':     (4.2, 3.1),
    'coche':    (4.5, 3.4),
    'manejar':  (3.8, 2.6),
    'rueda':    (4.8, 2.3),
    'velocidad':(3.5, 2.2),
    # Cluster frutas
    'banana':   (1.2, 6.8),
    'manzana':  (1.8, 7.2),
    'fruta':    (1.5, 6.3),
    'naranja':  (2.1, 6.6),
    # Cluster royalty (para analogía king-man+woman=queen)
    'rey':      (7.5, 7.0),
    'reina':    (7.0, 8.2),
    'hombre':   (8.2, 5.5),
    'mujer':    (7.7, 6.7),
    # Cluster finanzas
    'banco':    (6.0, 3.5),
    'dinero':   (6.5, 3.0),
    'crédito':  (5.8, 2.5),
}

colores = {
    'auto':'#7F77DD','coche':'#7F77DD','manejar':'#9F97ED',
    'rueda':'#9F97ED','velocidad':'#9F97ED',
    'banana':'#1D9E75','manzana':'#1D9E75','fruta':'#3DBE95','naranja':'#3DBE95',
    'rey':'#D85A30','reina':'#D85A30','hombre':'#E87A50','mujer':'#E87A50',
    'banco':'#D4A017','dinero':'#D4A017','crédito':'#E4B027',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Panel izquierdo: clusters semánticos ──────────────────────────────
ax = axes[0]
ax.set_facecolor('#F8F8FC')
ax.set_title('Espacio vectorial semántico (2D simplificado)', fontsize=12, pad=10)

for palabra, (x, y) in palabras.items():
    color = colores[palabra]
    ax.scatter(x, y, c=color, s=120, zorder=3, edgecolors='white', lw=1.5)
    ax.annotate(palabra, (x, y), xytext=(5, 5), textcoords='offset points',
                fontsize=9, fontweight='bold', color=color)

# Elipses de cluster
from matplotlib.patches import Ellipse
for cx, cy, w, h, color, label in [
    (4.2, 2.9, 1.8, 1.5, '#7F77DD', 'vehículos'),
    (1.7, 6.7, 1.5, 1.4, '#1D9E75', 'frutas'),
    (7.6, 6.9, 1.6, 2.0, '#D85A30', 'royalty'),
    (6.1, 3.0, 1.5, 1.2, '#D4A017', 'finanzas'),
]:
    e = Ellipse((cx, cy), w, h, angle=0, fill=True,
                facecolor=color, alpha=0.10, edgecolor=color, lw=1.5, linestyle='--')
    ax.add_patch(e)
    ax.annotate(label, (cx, cy - h/2 - 0.2), ha='center',
                fontsize=8, color=color, fontstyle='italic')

ax.set_xlim(0, 10); ax.set_ylim(1, 9)
ax.spines[['top','right']].set_visible(False)
ax.set_xlabel('Dimensión 1', fontsize=10)
ax.set_ylabel('Dimensión 2', fontsize=10)
ax.grid(True, alpha=0.2)

# ── Panel derecho: analogía king - man + woman ≈ queen ───────────────
ax2 = axes[1]
ax2.set_facecolor('#F8F8FC')
ax2.set_title('Analogía vectorial: rey − hombre + mujer ≈ reina', fontsize=12, pad=10)

puntos = {'rey':(7.5,7.0),'reina':(7.0,8.2),'hombre':(8.2,5.5),'mujer':(7.7,6.7)}
for p,(x,y) in puntos.items():
    ax2.scatter(x,y,c=colores[p],s=160,zorder=3,edgecolors='white',lw=2)
    ax2.annotate(p,(x,y),xytext=(8,6),textcoords='offset points',
                 fontsize=11,fontweight='bold',color=colores[p])

# Vectores de la analogía
O = np.array([8.2, 5.5])   # origen: hombre
A = np.array([7.5, 7.0])   # rey
B = np.array([7.7, 6.7])   # mujer
Q = np.array([7.0, 8.2])   # reina (resultado)
predicted = O + (A - O) + (B - O)  # hombre + (rey-hombre) + (mujer-hombre)

for start, end, color, label in [
    (O, A, '#D85A30', 'rey−hombre'),
    (O, B, '#E87A50', 'mujer−hombre'),
]:
    ax2.annotate('', xy=end, xytext=start,
                 arrowprops=dict(arrowstyle='->', color=color, lw=2))

ax2.scatter(*predicted, c='#7F77DD', s=220, zorder=4, marker='*',
            edgecolors='white', lw=1.5)
ax2.annotate('≈ reina
(predicho)', predicted,
             xytext=(8, 6), textcoords='offset points',
             fontsize=9, color='#7F77DD', fontweight='bold')

ax2.set_xlim(6.5, 9); ax2.set_ylim(4.8, 9)
ax2.spines[['top','right']].set_visible(False)
ax2.grid(True, alpha=0.2)
ax2.set_xlabel('Dimensión 1', fontsize=10)
ax2.set_ylabel('Dimensión 2', fontsize=10)

plt.tight_layout()
plt.savefig('embeddings_viz_clase1.png', dpi=150, bbox_inches='tight')
plt.show()

print('→ Panel izquierdo: palabras similares se agrupan naturalmente.')
print('→ Panel derecho:   las relaciones semánticas son operaciones vectoriales.')
print('   rey − hombre + mujer ≈ reina  (funciona en espacios de alta dimensión)')
print()
print('📖 Mikolov et al. (2013). https://arxiv.org/abs/1301.3781')

In [ ]:
# ✏️ EJERCICIO — experimentá con similitud coseno
# ¿Qué score tiene "auto" vs "coche"?
# ¿Y "banco" (dinero) vs "banco" (para sentarse)?
# ¿Dos oraciones en idiomas distintos se consideran similares?

frase_a = "..."  # reemplazá
frase_b = "..."  # reemplazá

# Completá acá
# score = cosine_similarity(modelo.encode([frase_a]), modelo.encode([frase_b]))[0][0]
# print(f'Score: {score:.4f}')

# Bloque 5
## Visualización con UMAP

---

*384 dimensiones → 2 dimensiones → clusters visibles*

## ¿Por qué visualizar?

Nuestros embeddings tienen **384 dimensiones**. Los humanos vemos 2 ó 3.

**UMAP** reduce la dimensionalidad preservando la estructura local:

- Puntos **cercanos en 384D** quedan **cercanos en 2D**
- Los clusters semánticos deberían ser visibles
- No es perfecto — UMAP es estocástico y el corpus es chico

```
[0.23, -0.14, 0.67, ... 384 números]  ──UMAP──▶  (x, y)
```

> 📖 *McInnes, L., Healy, J. & Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection. https://arxiv.org/abs/1802.03426*

In [ ]:
import umap

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=5,
    min_dist=0.3,
    random_state=42
)
embeddings_2d = reducer.fit_transform(embeddings)
print(f'✓ UMAP: {embeddings.shape[1]}D → 2D  |  Shape resultante: {embeddings_2d.shape}')

In [ ]:
colores_map = {
    'NLP':     '#7F77DD',
    'LLMs':    '#1D9E75',
    'RAG':     '#EF9F27',
    'Agentes': '#D85A30',
}

fig, ax = plt.subplots(figsize=(10, 6))

for categoria, color in colores_map.items():
    idx = [i for i, c in enumerate(CATEGORIAS) if c == categoria]
    ax.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1],
               c=color, label=categoria, s=100, alpha=0.85, edgecolors='white', lw=0.8)

for i, oracion in enumerate(CORPUS):
    ax.annotate(oracion[:32] + '...', (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                fontsize=6.5, alpha=0.7, xytext=(5, 5), textcoords='offset points')

ax.set_title('Clusters semánticos — corpus IA (UMAP 2D)', fontsize=13, pad=12)
ax.legend(loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('clusters_umap_clase1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ✏️ EJERCICIO — agregá tus propias oraciones
# 1. Agregá 2-3 oraciones sobre un tema nuevo (bases de datos, redes, seguridad...)
# 2. Regenerá los embeddings y el gráfico UMAP
# 3. ¿Se agrupan cerca de algún cluster existente o forman uno nuevo?

mis_oraciones_extra = [
    # "...",
    # "...",
]

if mis_oraciones_extra:
    corpus_ext = CORPUS + mis_oraciones_extra
    categorias_ext = CATEGORIAS + ['Nuevo tema'] * len(mis_oraciones_extra)
    emb_ext = modelo.encode(corpus_ext)
    emb_2d_ext = reducer.fit_transform(emb_ext)
    # Graficar (código similar al anterior)
    print('¡Agregá el código de graficación acá!')

## Lo que construimos hoy


---

## 📚 Bibliografía

### Libros de referencia

- Jurafsky, D. & Martin, J. H. (2024). *Speech and Language Processing* (3rd ed. draft). Stanford University.  
  🔗 https://web.stanford.edu/~jurafsky/slp3/

- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. MIT Press.  
  🔗 https://www.deeplearningbook.org/

- Tunstall, L., von Werra, L. & Wolf, T. (2022). *Natural Language Processing with Transformers*. O'Reilly Media.  
  🔗 https://www.oreilly.com/library/view/natural-language-processing/9781098136789/

### Papers

- Mikolov, T., Chen, K., Corrado, G. & Dean, J. (2013). Efficient Estimation of Word Representations in Vector Space. *ICLR 2013*.  
  🔗 https://arxiv.org/abs/1301.3781

- Sennrich, R., Haddow, B. & Birch, A. (2016). Neural Machine Translation of Rare Words with Subword Units. *ACL 2016*.  
  🔗 https://arxiv.org/abs/1508.07909

- Devlin, J., Chang, M., Lee, K. & Toutanova, K. (2018). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *NAACL 2019*.  
  🔗 https://arxiv.org/abs/1810.04805

- Reimers, N. & Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. *EMNLP 2019*.  
  🔗 https://arxiv.org/abs/1908.10084

- McInnes, L., Healy, J. & Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction.  
  🔗 https://arxiv.org/abs/1802.03426

## Clase 2 — ¿qué viene?

Hoy **representamos** texto. En Clase 2 vemos qué hace un LLM con esa representación para **generar** texto.

---

- **Tokenizers en modelos reales** — GPT-4o / Llama / BERT comparados con contexto completo
- **Arquitectura Transformer** — attention mechanism sin matemática pesada
- **Ciclo de vida del modelo** — pre-training → SFT → RLHF/DPO
- **Prompting como ingeniería** — zero-shot, few-shot, CoT, structured outputs